In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
pip install lightning

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.0/811.0 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 869.5/869.5 kB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 815.2/815.2 kB 52.4 MB/s eta 0:00:00


In [3]:
 pip install reformer_pytorch==1.4.4

  Preparing metadata (setup.py) ... done
  Created wheel for axial-positional-embedding: filename=axial_positional_embedding-0.2.1-py3-none-any.whl size=2888 sha256=af5fc733381cf5ad8e439d694ccdba6ad99d0cb56ceba5f302dcafcfeb68a8fa
  Stored in directory: /root/.cache/pip/wheels/b1/cb/39/7ce7ff2d2fd37cfe1fe7b3a3c43cf410632b2ad3b3f3986d73
Successfully built axial-positional-embedding


In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
import matplotlib.pyplot as plt
import tensorflow as tf
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
import os
np.random.seed(42)


In [27]:
import pandas as pd

datos = pd.read_csv('/content/drive/MyDrive/Tesis/Data/SNpollutionFinal.csv')

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [28]:
datos.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-01 00:00:00,129.0,-21,-11.0,1021.0,2,1.79
2010-01-01 01:00:00,129.0,-21,-12.0,1020.0,2,4.92
2010-01-01 02:00:00,129.0,-21,-11.0,1019.0,2,6.71
2010-01-01 03:00:00,129.0,-21,-14.0,1019.0,2,9.84
2010-01-01 04:00:00,129.0,-20,-12.0,1018.0,2,12.97


Se eliminan las primeras 24 filas debido a que estas contenían valores NAN en la columna pollution, y habían sido rellenadas con interpolación lineal.

In [29]:
datos = datos.drop(datos.index[:24])

datos

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,129.0,-16,-4.0,1020.0,1,1.79
2010-01-02 01:00:00,148.0,-15,-4.0,1020.0,1,2.68
2010-01-02 02:00:00,159.0,-11,-5.0,1021.0,1,3.57
2010-01-02 03:00:00,181.0,-7,-5.0,1022.0,1,5.36
2010-01-02 04:00:00,138.0,-7,-5.0,1022.0,1,6.25
...,...,...,...,...,...,...
2014-12-31 19:00:00,8.0,-23,-2.0,1034.0,2,231.97
2014-12-31 20:00:00,10.0,-22,-3.0,1034.0,2,237.78
2014-12-31 21:00:00,10.0,-22,-3.0,1034.0,2,242.70


In [30]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [31]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (30660, 6)
Las dimensiones de test son:  (8803, 6)
Las dimensiones de val son:  (4337, 6)


Se normalizan los datos

In [32]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [33]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [34]:
datosNormalizados.shape

(43800, 6)

In [35]:
datosNormalizados.head(10)


,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489
2010-01-02 05:00:00,0.098238,-0.586536,-1.430104,0.522567,-0.380944,-0.359017
2010-01-02 06:00:00,0.054349,-0.586536,-1.430104,0.619008,-0.380944,-0.323876
2010-01-02 07:00:00,0.262820,-0.586536,-1.349314,0.715448,-0.380944,-0.288735
2010-01-02 08:00:00,0.218931,-0.656257,-1.430104,0.715448,-0.380944,-0.253594


In [36]:
first_column = datosNormalizados.pop(datosNormalizados.columns[0])
datosNormalizados["pollution"] = first_column

In [37]:
datosNormalizados.head(10)

,dew,temp,press,wnd_dir,wnd_spd,pollution
date,,,,,,
2010-01-02 00:00:00,-1.214023,-1.268524,0.329687,-0.380944,-0.464048,0.317681
2010-01-02 01:00:00,-1.144302,-1.268524,0.329687,-0.380944,-0.446575,0.526152
2010-01-02 02:00:00,-0.865419,-1.349314,0.426127,-0.380944,-0.429103,0.646846
2010-01-02 03:00:00,-0.586536,-1.349314,0.522567,-0.380944,-0.393962,0.888234
2010-01-02 04:00:00,-0.586536,-1.349314,0.522567,-0.380944,-0.376489,0.416431
2010-01-02 05:00:00,-0.586536,-1.430104,0.522567,-0.380944,-0.359017,0.098238
2010-01-02 06:00:00,-0.586536,-1.430104,0.619008,-0.380944,-0.323876,0.054349
2010-01-02 07:00:00,-0.586536,-1.349314,0.715448,-0.380944,-0.288735,0.262820
2010-01-02 08:00:00,-0.656257,-1.430104,0.715448,-0.380944,-0.253594,0.218931


In [38]:
target = "pollution"
futuros = 1
pasados = 12

In [39]:
df = datosNormalizados.reset_index()
df['target'] = df[target].shift(-futuros)

df = df.dropna()

df.head(40)

,date,dew,temp,press,wnd_dir,wnd_spd,pollution,target
0,2010-01-02 00:00:00,-1.214023,-1.268524,0.329687,-0.380944,-0.464048,0.317681,0.526152
1,2010-01-02 01:00:00,-1.144302,-1.268524,0.329687,-0.380944,-0.446575,0.526152,0.646846
2,2010-01-02 02:00:00,-0.865419,-1.349314,0.426127,-0.380944,-0.429103,0.646846,0.888234
3,2010-01-02 03:00:00,-0.586536,-1.349314,0.522567,-0.380944,-0.393962,0.888234,0.416431
4,2010-01-02 04:00:00,-0.586536,-1.349314,0.522567,-0.380944,-0.376489,0.416431,0.098238
5,2010-01-02 05:00:00,-0.586536,-1.430104,0.522567,-0.380944,-0.359017,0.098238,0.054349
6,2010-01-02 06:00:00,-0.586536,-1.430104,0.619008,-0.380944,-0.323876,0.054349,0.262820
7,2010-01-02 07:00:00,-0.586536,-1.349314,0.715448,-0.380944,-0.288735,0.262820,0.218931
8,2010-01-02 08:00:00,-0.656257,-1.430104,0.715448,-0.380944,-0.253594,0.218931,0.350598
9,2010-01-02 09:00:00,-0.586536,-1.349314,0.811888,-0.380944,-0.218452,0.350598,0.438375


Se vuelve a dividir el dataframe

In [40]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(df, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (30659, 8)
Las dimensiones de test son:  (8803, 8)
Las dimensiones de val son:  (4337, 8)


Se crea un dataloader modificado

In [19]:
!git clone https://github.com/thuml/Time-Series-Library.git

Cloning into 'Time-Series-Library'...
remote: Enumerating objects: 1730, done.
remote: Counting objects: 100% (979/979), done.
remote: Compressing objects: 100% (261/261), done.
remote: Total 1730 (delta 792), reused 803 (delta 711), pack-reused 751 (from 1)
Receiving objects: 100% (1730/1730), 78.20 MiB | 23.28 MiB/s, done.
Resolving deltas: 100% (1179/1179), done.


In [20]:
import sys
sys.path.append('Time-Series-Library')

In [41]:
from models import Informer
from models import iTransformer
from utils.timefeatures import time_features

In [42]:
from torch.utils.data import Dataset


class datasetCustom(Dataset):
    def __init__(self, flag='train', size=None,
                 features='MS', target='OT', scale=False, timeenc=1, freq='h', seasonal_patterns=None, dataProc=None):
        # size [seq_len, label_len, pred_len]
        #self.args = args
        # info
        self.seq_len = size[0]
        self.label_len = size[1]
        self.pred_len = size[2]
        # init
        assert flag in ['train', 'test', 'val']
        type_map = {'train': 0, 'val': 1, 'test': 2}

        self.set_type = type_map[flag]

        self.features = features
        self.target = target
        #self.scale = scale
        self.timeenc = timeenc
        self.freq = freq

        #self.root_path = root_path
        self.dataProc = dataProc
        self.__read_data__()

    def __read_data__(self):
        #self.scaler = StandardScaler()
        df_raw = self.dataProc

        '''
        df_raw.columns: ['date', ...(other features), target feature]
        '''

        self.targetSet = df_raw[df_raw.columns[1:]].values
        cols = list(df_raw.columns)
        cols.remove("target")
        cols.remove('date')

        df_raw = df_raw[['date'] + cols] #+ ["target"]]

        '''
        num_train = int(len(df_raw) * 0.7)
        num_test = int(len(df_raw) * 0.2)
        num_vali = len(df_raw) - num_train - num_test
        border1s = [0, num_train - self.seq_len, len(df_raw) - num_test - self.seq_len]
        border2s = [num_train, num_train + num_vali, len(df_raw)]
        border1 = border1s[self.set_type]
        border2 = border2s[self.set_type]
        '''
        if self.features == 'M' or self.features == 'MS':
            cols_data = df_raw.columns[1:]
            df_data = df_raw[cols_data]
        elif self.features == 'S':
            df_data = df_raw[[self.target]]
        '''
        if self.scale:
            train_data = df_data[border1s[0]:border2s[0]]
            self.scaler.fit(train_data.values)
            data = self.scaler.transform(df_data.values)
        else:'''
        data = df_data.values

        df_stamp = df_raw[['date']]  #[border1:border2]
        df_stamp['date'] = pd.to_datetime(df_stamp.date)
        #print(type(df_stamp))
        if self.timeenc == 0:
            df_stamp['year'] = df_stamp.date.apply(lambda row: row.year, 1)
            df_stamp['month'] = df_stamp.date.apply(lambda row: row.month, 1)
            df_stamp['day'] = df_stamp.date.apply(lambda row: row.day, 1)
            df_stamp['hour'] = df_stamp.date.apply(lambda row: row.hour, 1)
            data_stamp = df_stamp.drop(['date'], axis=1).values
        elif self.timeenc == 1:
            data_stamp = time_features(pd.to_datetime(df_stamp['date'].values), freq=self.freq)
            data_stamp = data_stamp.transpose(1, 0)

        self.data_x = data
        self.data_y = data
        #print("dimensiones de x:", self.data_x.shape)
        #print("dimensiones de y:", self.data_y.shape)
        '''
        if self.set_type == 0 and self.args.augmentation_ratio > 0:
            self.data_x, self.data_y, augmentation_tags = run_augmentation_single(self.data_x, self.data_y, self.args)
        '''
        self.data_stamp = data_stamp

    def __getitem__(self, index):
        s_begin = index
        s_end = s_begin + self.seq_len
        r_begin = s_end - self.label_len
        r_end = r_begin + self.label_len + self.pred_len

        seq_x = self.data_x[s_begin:s_end]
        seq_y = self.data_y[r_begin:r_end]
        #print("dimensiones de x en getItem:", seq_x.shape)
        #print("dimensiones de y en getItem:", seq_y.shape)
        seq_x_mark = self.data_stamp[s_begin:s_end]
        seq_y_mark = self.data_stamp[r_begin:r_end]
        seq_target = self.targetSet[r_begin:r_end]

        return seq_x, seq_y, seq_x_mark, seq_y_mark, seq_target

    def __len__(self):
        return len(self.data_x) - self.seq_len - self.pred_len + 1


In [43]:
features = 'MS'
target = 'pollution'
freq = 'h'

In [44]:
seq_len = 12 # input sequence length of Informer encoder
label_len = 12 # start token length of Informer decoder
pred_len = 1 # prediction sequence length
timeenc=0

In [45]:
from torch.utils.data import DataLoader

trainDataset = datasetCustom(
            size=[seq_len, label_len, pred_len],
            features=features,
            target=target,
            timeenc=timeenc,
            freq=freq,
            dataProc=train
        )
testDataset = datasetCustom(
            size=[seq_len, label_len, pred_len],
            features=features,
            target=target,
            timeenc=timeenc,
            freq=freq,
            dataProc=test
        )
valDataset = datasetCustom(
            size=[seq_len, label_len, pred_len],
            features=features,
            target=target,
            timeenc=timeenc,
            freq=freq,
            dataProc=val
        )

<ipython-input-42-a9425b45dae7>:67: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_stamp['date'] = pd.to_datetime(df_stamp.date)
<ipython-input-42-a9425b45dae7>:70: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  df_stamp['year'] = df_stamp.date.apply(lambda row: row.year, 1)
<ipython-input-42-a9425b45dae7>:70: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_stamp['year'] = 

In [46]:
a = trainDataset.data_y
a

array([[-1.2140229 , -1.26852411,  0.32968671, -0.38094383, -0.46404777,
         0.31768099],
       [-1.14430217, -1.26852411,  0.32968671, -0.38094383, -0.44657536,
         0.52615226],
       [-0.86541928, -1.34931411,  0.42612698, -0.38094383, -0.42910295,
         0.64684616],
       ...,
       [ 1.22620242,  1.23596569, -1.98487987,  0.69453847,  0.17654188,
        -0.93314668],
       [ 1.22620242,  1.31675569, -1.98487987,  0.69453847,  0.2379898 ,
        -0.93314668],
       [ 1.08676097,  1.39754568, -1.98487987,  0.69453847,  0.35205128,
        -0.92217451]])

In [47]:
import torch
import torch.nn as nn
import torch.optim as optim
import pytorch_lightning as pl

class InformerLightning(pl.LightningModule):
    def __init__(self, model, args):
        super(InformerLightning, self).__init__()
        self.model = model  # El modelo de Informer
        self.args = args  # Argumentos como epochs

    def forward(self, batch_x, batch_x_mark, dec_inp, batch_y_mark):
        return self.model(batch_x, batch_x_mark, dec_inp, batch_y_mark)

    def training_step(self, batch, batch_idx):
        batch_x, batch_y, batch_x_mark, batch_y_mark, seq_target = batch
        batch_x = batch_x.float()
        batch_y = batch_y.float()
        batch_x_mark = batch_x_mark.float()
        batch_y_mark = batch_y_mark.float()
        seq_target = seq_target.float()

        # Input del decoder
        dec_inp = torch.zeros_like(batch_y[:, -self.args.pred_len:, :]).float()
        dec_inp = torch.cat([batch_y[:, :self.args.label_len, :], dec_inp], dim=1).float()

        # Forward pass
        outputs = self(batch_x, batch_x_mark, dec_inp, batch_y_mark)
        #print("shape de output: ", outputs.shape)
        f_dim = -1 if self.args.features == 'MS' else 0
        outputs = outputs[:, -self.args.pred_len, f_dim:]
        real = seq_target[:, -self.args.pred_len-1, f_dim:]
        loss = nn.MSELoss()(real, outputs)

        self.log('train_loss', loss)
        return loss

    def validation_step(self, batch, batch_idx):
        batch_x, batch_y, batch_x_mark, batch_y_mark, seq_target = batch
        batch_x = batch_x.float()
        batch_y = batch_y.float()
        batch_x_mark = batch_x_mark.float()
        batch_y_mark = batch_y_mark.float()
        seq_target = seq_target.float()

        # Input del decoder
        dec_inp = torch.zeros_like(batch_y[:, -self.args.pred_len:, :]).float()
        dec_inp = torch.cat([batch_y[:, :self.args.label_len, :], dec_inp], dim=1).float()

        # Forward pass
        outputs = self(batch_x, batch_x_mark, dec_inp, batch_y_mark)
        f_dim = -1 if self.args.features == 'MS' else 0
        outputs = outputs[:, -self.args.pred_len, f_dim:]
        real = seq_target[:, -self.args.pred_len-1, f_dim:]
        loss = nn.MSELoss()(real, outputs)

        self.log('val_loss', loss)
        return loss

    def configure_optimizers(self):
        optimizer = self._select_optimizer()
        return optimizer

    def _select_optimizer(self):
        return optim.Adam(self.parameters(), lr=float(self.args.learning_rate))

    @classmethod
    def load_from_checkpoint(cls, checkpoint_path, model, conf, *args, **kwargs):
        # Cargar el estado del modelo desde el checkpoint
        checkpoint = torch.load(checkpoint_path)
        # Crear una instancia del modelo
        instance = cls(model, args=conf, *args, **kwargs)
        instance.load_state_dict(checkpoint['state_dict'])
        return instance


In [48]:
entradas = 6
frecuencia = 'h'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [49]:
space = {
    'heads': hp.choice('heads', [2 ** i for i in range(1, 4)]),
    'e_layers': hp.quniform('e_layers', 1, 4, 1),
    'dropout':hp.quniform('dropout', 0, 0.5, 0.1),
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)]),
    'd_model': hp.choice('d_model', [2 ** i for i in range(6, 10)]),
    'activation': hp.choice('activation', ["relu", "gelu"]),
}

In [50]:
class Configs:
    def __init__(self, hyperparams):
        # Parámetros generales
        self.task_name = 'long_term_forecast'
        #self.enc_in = entradas
        #self.dec_in = entradas
        #self.c_out = 1
        self.seq_len = pasados  # entrada
        self.label_len = label_len  # Definir este valor antes
        self.pred_len = 1  # Longitud de la predicción

        # Hiperparámetros que vienen del espacio de búsqueda
        self.d_model = int(hyperparams['d_model'])
        self.d_ff = self.d_model * 4
        self.n_heads = int(hyperparams['heads'])
        self.e_layers = int(hyperparams['e_layers'])
        self.factor = 5
        self.dropout = float(hyperparams['dropout'])
        self.learning_rate = float(hyperparams['learning_rate'])

        # Parámetros fijos o ajustables
        self.embed = 'fixed'
        self.freq = frecuencia
        self.activation = hyperparams['activation']
        self.features = "MS"
        #self.distil = True
        self.task_name = 'long_term_forecast'


In [51]:
def objective(params):
    print(params)
    early_stop_callback = EarlyStopping(
      monitor='val_loss',
      min_delta=0.00,
      patience=15,
      verbose=True,
      mode='min'
    )

    checkpoint_callback = ModelCheckpoint(
    monitor='val_loss',
    dirpath='checkpoints',
    filename='best-checkpoint',
    save_top_k=1,
    mode='min',
    )

    configs = Configs(params)


    model = iTransformer.Model(configs)
    informerModel = InformerLightning(model, configs)


    train_loader = DataLoader(trainDataset, batch_size=int(params["batch"]), shuffle=False)
    val_loader = DataLoader(testDataset, batch_size=int(params["batch"]), shuffle=False)


    trainer = pl.Trainer(
        callbacks=[early_stop_callback, checkpoint_callback],
        max_epochs=128,
        accelerator='gpu',
        enable_progress_bar=True,
        logger=False
    )


    trainer.fit(informerModel, train_loader, val_dataloaders=val_loader)

    bestModel = InformerLightning.load_from_checkpoint(trainer.checkpoint_callback.best_model_path, model=model, conf=configs)

    informerModel.eval()
    preds = []
    trues = []

    test_loader = DataLoader(valDataset, batch_size=32, shuffle=False)

    with torch.no_grad():
        for i, (batch_x, batch_y, batch_x_mark, batch_y_mark, seq_target) in enumerate(test_loader):
            batch_x = batch_x.float()
            batch_y = batch_y.float()
            batch_x_mark = batch_x_mark.float()
            batch_y_mark = batch_y_mark.float()
            seq_target = seq_target.float()


            dec_inp = torch.zeros_like(batch_y[:, -configs.pred_len:, :]).float()
            dec_inp = torch.cat([batch_y[:, :configs.label_len, :], dec_inp], dim=1).float().to(device)

            outputs = bestModel(batch_x, batch_x_mark, dec_inp, batch_y_mark)
            outputs = outputs[:, -configs.pred_len, -1:]
            batch_y = seq_target[:, -configs.pred_len-1, -1:]


            preds.append(outputs)
            trues.append(batch_y)

    preds = np.concatenate(preds, axis=0)
    trues = np.concatenate(trues, axis=0)

    preds = preds.reshape(-1, preds.shape[-1])
    trues = trues.reshape(-1, trues.shape[-1])

    print("preds shape:", preds.shape)
    print("trues shape:", trues.shape)
    loss = mean_squared_error(trues, preds)

    return {'loss': loss, 'status': STATUS_OK}

In [52]:
import pytorch_lightning as pl

# Fijar la semilla para la reproducibilidad
pl.seed_everything(42)

INFO:lightning_fabric.utilities.seed:Seed set to 42


42

In [53]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=12, trials=trials, rstate=np.random.default_rng(42))

print(best)

{'activation': 'gelu', 'batch': 32, 'd_model': 256, 'dropout': 0.30000000000000004, 'e_layers': 4.0, 'heads': 4, 'learning_rate': 0.0017973043712331907}
  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name  | Type  | Params | Mode 
----------------------------------------
0 | model | Model | 3.2 M  | train
----------------------------------------
3.2 M     Trainable params
0         Non-trainable params
3.2 M     Total params
12.653    Total estimated model params size (MB)
60        Modules in train mode
0         Modules in eval mode


Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/usr/local/lib/python3.10/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.

/usr/local/lib/python3.10/dist-packages/pytorch_lightning/trainer/connectors/data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=7` in the `DataLoader` to improve performance.



Training: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

INFO:pytorch_lightning.callbacks.early_stopping:Monitored metric val_loss did not improve in the last 15 records. Best score: 0.070. Signaling Trainer to stop.
<ipython-input-47-051e6e8a8c6d>:70: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an is

preds shape:
(4325, 1)
trues shape:
(4325, 1)
100%|██████████| 12/12 [2:49:36<00:00, 848.04s/trial, best loss: 0.03827422484755516]
{'activation': 0, 'batch': 0, 'd_model': 0, 'dropout': 0.2, 'e_layers': 2.0, 'heads': 1, 'learning_rate': 0.00010271054144739545}


In [54]:
print(best)

{'activation': 0, 'batch': 0, 'd_model': 0, 'dropout': 0.2, 'e_layers': 2.0, 'heads': 1, 'learning_rate': 0.00010271054144739545}
